# 02. Dog Detection and Cropping YOLO — `proyecto_integrador_v2`

Este notebook corresponde al **Paso 02** del pipeline `proyecto_integrador_v2`.

## Objetivo

Detectar perros en las imágenes curadas del Paso 01 y generar **crops del perro** que serán usados posteriormente para extracción de embeddings y búsqueda visual.

En esta versión del proyecto, el objetivo principal no es clasificar la raza, sino construir una base visual para re-identificación de perros perdidos/encontrados. Por eso, este paso se enfoca en:

- detectar correctamente al perro,
- seleccionar la mejor detección,
- generar un crop útil,
- conservar metadata de la imagen original,
- guardar indicadores de detección,
- marcar casos que requieren revisión.

## Entrada principal

Este notebook usa como entrada el reporte generado por el Paso 01:

```text
curated_data/quality_reports/step01_image_quality_report.csv
```

La imagen principal para YOLO será:

```text
curated_image_path
```

No se usa `model_ready_224` como fuente principal para YOLO, porque esa versión ya tiene padding y está pensada para modelos de entrada fija, no para detección.

## Salidas del Paso 02

El notebook generará:

```text
processed_data/dog_crops/
processed_data/metadata/step02_detection_report.csv
reports/tables/step02_detection_indicators.csv
reports/figures/step02_detection_samples.png
```

Los crops generados en este paso serán la entrada principal para:

```text
03_Dog_Visual_Embeddings.ipynb
```

In [ ]:
# 0. Instalación de dependencias

!pip install -q ultralytics opencv-python pillow pandas numpy matplotlib tqdm

In [ ]:
# 1. Imports y configuración general

from pathlib import Path
import json
import time
import math
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm

from ultralytics import YOLO

pd.set_option("display.max_columns", 200)

SEED = 42
np.random.seed(SEED)

print("OpenCV:", cv2.__version__)

In [ ]:
# 2. Montar Google Drive

from google.colab import drive

if Path("/content/drive/MyDrive").exists():
    print("Google Drive ya está montado.")
else:
    drive.mount("/content/drive")

In [ ]:
# 3. Rutas del proyecto

PROJECT_ROOT = Path("/content/drive/MyDrive/proyecto_integrador_v2")
CONFIG_PATH = PROJECT_ROOT / "project_config.json"

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = json.load(f)
else:
    config = {}

CURATED_DATA_PATH = PROJECT_ROOT / "curated_data"
QUALITY_REPORTS_PATH = CURATED_DATA_PATH / "quality_reports"
STEP01_QUALITY_REPORT_PATH = QUALITY_REPORTS_PATH / "step01_image_quality_report.csv"

PROCESSED_DATA_PATH = PROJECT_ROOT / "processed_data"
DOG_CROPS_PATH = PROCESSED_DATA_PATH / "dog_crops"
METADATA_PATH = PROCESSED_DATA_PATH / "metadata"

REPORTS_PATH = PROJECT_ROOT / "reports"
FIGURES_PATH = REPORTS_PATH / "figures"
TABLES_PATH = REPORTS_PATH / "tables"

MODELS_PATH = PROJECT_ROOT / "models"
YOLO_MODELS_PATH = MODELS_PATH / "yolo"

for p in [DOG_CROPS_PATH, METADATA_PATH, REPORTS_PATH, FIGURES_PATH, TABLES_PATH, YOLO_MODELS_PATH]:
    p.mkdir(parents=True, exist_ok=True)

detector_config = config.get("detector", {})

YOLO_MODEL_NAME = detector_config.get("default_model", "yolo26s.pt")
FALLBACK_YOLO_MODEL_NAME = detector_config.get("fallback_model", "yolo11s.pt")
CONF_THRESHOLD = float(detector_config.get("confidence_threshold", 0.25))
CROP_MARGIN = float(detector_config.get("crop_margin", 0.15))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("STEP01_QUALITY_REPORT_PATH:", STEP01_QUALITY_REPORT_PATH)
print("DOG_CROPS_PATH:", DOG_CROPS_PATH)
print("YOLO_MODEL_NAME:", YOLO_MODEL_NAME)
print("CONF_THRESHOLD:", CONF_THRESHOLD)
print("CROP_MARGIN:", CROP_MARGIN)

In [ ]:
# 4. Cargar reporte del Paso 01

if not STEP01_QUALITY_REPORT_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró {STEP01_QUALITY_REPORT_PATH}. "
        "Ejecuta primero el Paso 01."
    )

quality_df = pd.read_csv(STEP01_QUALITY_REPORT_PATH)

print("Registros cargados del Paso 01:", len(quality_df))
print("Columnas:")
print(quality_df.columns.tolist())

display(quality_df.head())

## Selección de imágenes para YOLO

Para detección usamos `curated_image_path`, porque conserva mejor la estructura original de la imagen.  
Las imágenes `needs_review` no se eliminan automáticamente; se procesan y se marcan en metadata para análisis posterior.

Esto es importante porque, en un sistema real de perros perdidos/encontrados, una imagen difícil todavía podría contener información útil.

In [ ]:
# 5. Preparar dataframe de detección

required_cols = ["image_path", "curated_image_path", "read_status", "quality_label"]
missing = [c for c in required_cols if c not in quality_df.columns]

if missing:
    raise ValueError(f"Faltan columnas necesarias del Paso 01: {missing}")

detection_input_df = quality_df[
    quality_df["read_status"] == "ok"
].copy().reset_index(drop=True)

detection_input_df["detection_input_path"] = detection_input_df["curated_image_path"]

missing_curated = detection_input_df["detection_input_path"].isna() | (
    detection_input_df["detection_input_path"].astype(str).str.len() == 0
)

detection_input_df.loc[missing_curated, "detection_input_path"] = detection_input_df.loc[
    missing_curated, "image_path"
]

detection_input_df["detection_input_exists"] = detection_input_df["detection_input_path"].apply(
    lambda p: Path(str(p)).exists()
)

print("Imágenes listas para detección:", len(detection_input_df))
print("Existencia de archivos:")
print(detection_input_df["detection_input_exists"].value_counts(dropna=False))

detection_input_df = (
    detection_input_df[detection_input_df["detection_input_exists"]]
    .drop(columns=["detection_input_exists"])
    .reset_index(drop=True)
)

print("Imágenes válidas para YOLO:", len(detection_input_df))
display(detection_input_df[["source_type", "quality_label", "detection_input_path"]].head())

In [ ]:
# 6. Cargar modelo YOLO

def load_yolo_detector(primary_model: str, fallback_model: str):
    """Carga YOLO. Si el modelo principal falla, usa fallback."""
    try:
        print(f"Intentando cargar detector principal: {primary_model}")
        model = YOLO(primary_model)
        selected_model = primary_model
    except Exception as e:
        print("No se pudo cargar el detector principal.")
        print("Error:", e)
        print(f"Usando detector fallback: {fallback_model}")
        model = YOLO(fallback_model)
        selected_model = fallback_model

    return model, selected_model


yolo_detector, selected_yolo_model = load_yolo_detector(
    YOLO_MODEL_NAME,
    FALLBACK_YOLO_MODEL_NAME
)

dog_class_id = None

for class_id, class_name in yolo_detector.names.items():
    if str(class_name).lower() == "dog":
        dog_class_id = int(class_id)
        break

if dog_class_id is None:
    raise ValueError("No se encontró la clase 'dog' en el modelo YOLO seleccionado.")

print("Modelo YOLO seleccionado:", selected_yolo_model)
print("ID clase dog:", dog_class_id)
print("Clases disponibles:", yolo_detector.names)

In [ ]:
# 7. Funciones de detección y crop

def detect_dogs_yolo(image_path, model, dog_class_id, conf_threshold=0.25):
    """Detecta perros en una imagen usando YOLO."""
    results = model(str(image_path), conf=conf_threshold, verbose=False)

    detections = []

    for result in results:
        boxes = result.boxes

        if boxes is None:
            continue

        for box in boxes:
            class_id = int(box.cls[0])
            confidence = float(box.conf[0])

            if class_id == dog_class_id:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)

                w = max(0, x2 - x1)
                h = max(0, y2 - y1)
                area = int(w * h)

                detections.append({
                    "class_id": class_id,
                    "class_name": "dog",
                    "confidence": confidence,
                    "x1": int(x1),
                    "y1": int(y1),
                    "x2": int(x2),
                    "y2": int(y2),
                    "box_width": int(w),
                    "box_height": int(h),
                    "area": area
                })

    return detections


def select_best_detection(detections):
    """Selecciona la detección principal usando área y confianza."""
    if len(detections) == 0:
        return None

    return sorted(
        detections,
        key=lambda d: (d["area"], d["confidence"]),
        reverse=True
    )[0]


def crop_detection(image_path, detection, output_path, margin=0.15):
    """Genera crop con margen alrededor de la detección."""
    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        return None, "image_not_readable"

    img_h, img_w = image_bgr.shape[:2]

    x1, y1, x2, y2 = detection["x1"], detection["y1"], detection["x2"], detection["y2"]

    box_w = x2 - x1
    box_h = y2 - y1

    mx = int(box_w * margin)
    my = int(box_h * margin)

    x1m = max(0, x1 - mx)
    y1m = max(0, y1 - my)
    x2m = min(img_w, x2 + mx)
    y2m = min(img_h, y2 + my)

    if x2m <= x1m or y2m <= y1m:
        return None, "invalid_crop_coordinates"

    crop = image_bgr[y1m:y2m, x1m:x2m]

    output_path.parent.mkdir(parents=True, exist_ok=True)

    ok = cv2.imwrite(str(output_path), crop)

    if not ok:
        return None, "crop_write_failed"

    crop_h, crop_w = crop.shape[:2]

    crop_info = {
        "crop_path": str(output_path),
        "crop_width": int(crop_w),
        "crop_height": int(crop_h),
        "crop_x1": int(x1m),
        "crop_y1": int(y1m),
        "crop_x2": int(x2m),
        "crop_y2": int(y2m)
    }

    return crop_info, None


def build_crop_output_path(row, input_path):
    """Construye ruta de salida para crop conservando estructura relativa."""
    source_type = row.get("source_type", "unknown")
    relative_path = row.get("relative_path", None)

    if pd.isna(relative_path) or relative_path is None:
        input_path = Path(input_path)
        relative_path = Path(source_type) / input_path.name
    else:
        relative_path = Path(str(relative_path))

    output_path = DOG_CROPS_PATH / source_type / relative_path

    # Forzar extensión jpg para los crops
    output_path = output_path.with_suffix(".jpg")

    return output_path

## Ejecución de detección

Cada imagen se procesa con YOLO.  
Si se detecta más de un perro, se selecciona la detección con mayor área y confianza.

Esto es razonable para una primera versión del sistema, pero los casos con múltiples perros se marcarán para revisión porque en un escenario real podrían requerir manejo especial.

In [ ]:
# 8. Ejecutar detección y generación de crops

detection_records = []

start = time.time()

for _, row in tqdm(detection_input_df.iterrows(), total=len(detection_input_df)):
    input_path = Path(row["detection_input_path"])

    base_record = row.to_dict()
    base_record.update({
        "selected_yolo_model": selected_yolo_model,
        "conf_threshold": CONF_THRESHOLD,
        "crop_margin": CROP_MARGIN,
        "dog_detected": False,
        "num_dog_detections": 0,
        "best_confidence": np.nan,
        "best_area": np.nan,
        "best_x1": np.nan,
        "best_y1": np.nan,
        "best_x2": np.nan,
        "best_y2": np.nan,
        "crop_status": "not_created",
        "crop_error": None,
        "crop_path": None,
        "crop_width": np.nan,
        "crop_height": np.nan,
        "needs_review_detection": False,
        "detection_review_reason": "none"
    })

    try:
        detections = detect_dogs_yolo(
            input_path,
            yolo_detector,
            dog_class_id,
            conf_threshold=CONF_THRESHOLD
        )

        base_record["num_dog_detections"] = len(detections)

        best_detection = select_best_detection(detections)

        if best_detection is None:
            base_record["dog_detected"] = False
            base_record["crop_status"] = "no_dog_detected"
            base_record["needs_review_detection"] = True
            base_record["detection_review_reason"] = "no_dog_detected"
            detection_records.append(base_record)
            continue

        base_record["dog_detected"] = True
        base_record["best_confidence"] = best_detection["confidence"]
        base_record["best_area"] = best_detection["area"]
        base_record["best_x1"] = best_detection["x1"]
        base_record["best_y1"] = best_detection["y1"]
        base_record["best_x2"] = best_detection["x2"]
        base_record["best_y2"] = best_detection["y2"]

        if len(detections) > 1:
            base_record["needs_review_detection"] = True
            base_record["detection_review_reason"] = "multiple_dogs_detected"

        crop_output_path = build_crop_output_path(row, input_path)

        crop_info, crop_error = crop_detection(
            input_path,
            best_detection,
            crop_output_path,
            margin=CROP_MARGIN
        )

        if crop_error is not None:
            base_record["crop_status"] = "error"
            base_record["crop_error"] = crop_error
            base_record["needs_review_detection"] = True

            if base_record["detection_review_reason"] == "none":
                base_record["detection_review_reason"] = crop_error
        else:
            base_record["crop_status"] = "ok"
            base_record.update(crop_info)

    except Exception as e:
        base_record["crop_status"] = "error"
        base_record["crop_error"] = str(e)
        base_record["needs_review_detection"] = True
        base_record["detection_review_reason"] = "exception"

    detection_records.append(base_record)

detection_df = pd.DataFrame(detection_records)

elapsed = (time.time() - start) / 60

print("Detección terminada.")
print("Tiempo:", round(elapsed, 2), "minutos")
print("Registros:", len(detection_df))

print("Dog detected:")
print(detection_df["dog_detected"].value_counts(dropna=False))

print("Crop status:")
print(detection_df["crop_status"].value_counts(dropna=False))

display(detection_df.head())

In [ ]:
# 9. Guardar reportes de detección

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

DETECTION_REPORT_PATH = METADATA_PATH / "step02_detection_report.csv"
DETECTION_REPORT_TIMESTAMPED_PATH = METADATA_PATH / f"step02_detection_report_{timestamp}.csv"

detection_df.to_csv(DETECTION_REPORT_PATH, index=False)
detection_df.to_csv(DETECTION_REPORT_TIMESTAMPED_PATH, index=False)

# Copia en reports/tables
detection_df.to_csv(TABLES_PATH / "step02_detection_report.csv", index=False)

print("Reporte de detección guardado en:", DETECTION_REPORT_PATH)
print("Reporte con timestamp guardado en:", DETECTION_REPORT_TIMESTAMPED_PATH)

In [ ]:
# 10. Indicadores finales del Paso 02

total_images = len(detection_df)
detected_images = int(detection_df["dog_detected"].sum()) if total_images > 0 else 0
not_detected_images = int((~detection_df["dog_detected"]).sum()) if total_images > 0 else 0

crops_ok = int((detection_df["crop_status"] == "ok").sum()) if total_images > 0 else 0
crop_errors = int((detection_df["crop_status"] == "error").sum()) if total_images > 0 else 0
no_dog_detected = int((detection_df["crop_status"] == "no_dog_detected").sum()) if total_images > 0 else 0

multiple_dogs = int((detection_df["num_dog_detections"] > 1).sum()) if total_images > 0 else 0
needs_review = int(detection_df["needs_review_detection"].sum()) if total_images > 0 else 0

detection_rate = detected_images / total_images * 100 if total_images > 0 else np.nan
crop_success_rate = crops_ok / total_images * 100 if total_images > 0 else np.nan

avg_yolo_confidence = float(detection_df["best_confidence"].dropna().mean()) if detected_images > 0 else np.nan
avg_num_detections = float(detection_df["num_dog_detections"].mean()) if total_images > 0 else np.nan

step02_indicators_df = pd.DataFrame([
    {"section": "input", "indicator": "total_images_evaluated", "value": total_images},
    {"section": "detection", "indicator": "images_with_dog_detected", "value": detected_images},
    {"section": "detection", "indicator": "images_without_dog_detected", "value": not_detected_images},
    {"section": "detection", "indicator": "dog_detection_rate_percent", "value": detection_rate},
    {"section": "detection", "indicator": "avg_yolo_confidence", "value": avg_yolo_confidence},
    {"section": "detection", "indicator": "avg_num_detections", "value": avg_num_detections},
    {"section": "detection", "indicator": "multiple_dogs_detected_cases", "value": multiple_dogs},
    {"section": "crop", "indicator": "crops_created_ok", "value": crops_ok},
    {"section": "crop", "indicator": "crop_errors", "value": crop_errors},
    {"section": "crop", "indicator": "no_dog_detected", "value": no_dog_detected},
    {"section": "crop", "indicator": "crop_success_rate_percent", "value": crop_success_rate},
    {"section": "review", "indicator": "needs_review_detection_cases", "value": needs_review},
    {"section": "model", "indicator": "selected_yolo_model", "value": selected_yolo_model},
    {"section": "model", "indicator": "confidence_threshold", "value": CONF_THRESHOLD},
    {"section": "model", "indicator": "crop_margin", "value": CROP_MARGIN},
])

STEP02_INDICATORS_PATH = TABLES_PATH / "step02_detection_indicators.csv"
step02_indicators_df.to_csv(STEP02_INDICATORS_PATH, index=False)

display(step02_indicators_df)
print("Indicadores guardados en:", STEP02_INDICATORS_PATH)

In [ ]:
# 11. Resumen por fuente y calidad

if len(detection_df) > 0:
    source_detection_summary_df = (
        detection_df
        .groupby(["source_type", "quality_label", "crop_status"], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values(["source_type", "quality_label", "crop_status"])
    )

    SOURCE_DETECTION_SUMMARY_PATH = TABLES_PATH / "step02_source_detection_summary.csv"
    source_detection_summary_df.to_csv(SOURCE_DETECTION_SUMMARY_PATH, index=False)

    display(source_detection_summary_df)
    print("Resumen guardado en:", SOURCE_DETECTION_SUMMARY_PATH)
else:
    print("No hay registros para resumir.")

In [ ]:
# 12. Visualización de muestras de crops

def show_crop_samples(df, n=10):
    ok_crops_df = df[df["crop_status"] == "ok"].copy()

    if len(ok_crops_df) == 0:
        print("No hay crops para mostrar.")
        return

    sample_df = ok_crops_df.sample(min(n, len(ok_crops_df)), random_state=SEED)

    cols = min(5, len(sample_df))
    rows = math.ceil(len(sample_df) / cols)

    plt.figure(figsize=(4 * cols, 4 * rows))

    for i, (_, row) in enumerate(sample_df.iterrows()):
        crop_path = row["crop_path"]

        try:
            img = Image.open(crop_path).convert("RGB")
            plt.subplot(rows, cols, i + 1)
            plt.imshow(img)
            title = f"{row['source_type']}\\nconf={row['best_confidence']:.2f}"
            plt.title(title, fontsize=9)
            plt.axis("off")
        except Exception as e:
            print("Error mostrando crop:", crop_path, e)

    plt.tight_layout()

    sample_path = FIGURES_PATH / "step02_detection_crop_samples.png"
    plt.savefig(sample_path, dpi=150)
    plt.show()

    print("Muestra guardada en:", sample_path)


show_crop_samples(detection_df, n=10)

In [ ]:
# 13. Visualización de casos para revisión

def show_review_samples(df, n=8):
    review_df = df[df["needs_review_detection"] == True].copy()

    if len(review_df) == 0:
        print("No hay casos marcados para revisión.")
        return

    sample_df = review_df.sample(min(n, len(review_df)), random_state=SEED)

    cols = min(4, len(sample_df))
    rows = math.ceil(len(sample_df) / cols)

    plt.figure(figsize=(4 * cols, 4 * rows))

    for i, (_, row) in enumerate(sample_df.iterrows()):
        img_path = row["detection_input_path"]

        try:
            img = Image.open(img_path).convert("RGB")
            plt.subplot(rows, cols, i + 1)
            plt.imshow(img)
            title = f"{row['detection_review_reason']}\\nstatus={row['crop_status']}"
            plt.title(title, fontsize=8)
            plt.axis("off")
        except Exception as e:
            print("Error mostrando:", img_path, e)

    plt.tight_layout()

    review_path = FIGURES_PATH / "step02_detection_review_samples.png"
    plt.savefig(review_path, dpi=150)
    plt.show()

    print("Muestra de revisión guardada en:", review_path)


show_review_samples(detection_df, n=8)

# Análisis de métricas del Paso 02

Este paso evalúa si el sistema puede localizar correctamente al perro dentro de cada imagen y generar un crop útil para los embeddings.

Las métricas principales son:

- **dog_detection_rate_percent**: porcentaje de imágenes donde YOLO detectó un perro.
- **avg_yolo_confidence**: confianza promedio de la detección principal.
- **crops_created_ok**: número de crops generados correctamente.
- **crop_success_rate_percent**: porcentaje de imágenes que terminaron con crop válido.
- **multiple_dogs_detected_cases**: casos donde se detectó más de un perro.
- **needs_review_detection_cases**: imágenes que requieren revisión por falta de detección, múltiples perros o error de crop.

Para el objetivo de perros perdidos/encontrados, los crops son más importantes que la clasificación de raza, porque los siguientes pasos usarán estos recortes para generar embeddings visuales y buscar coincidencias.

# Comentarios finale sobre el Paso 02

El Paso 02 transforma las imágenes curadas del Paso 01 en regiones visuales centradas en el perro.

Este paso es fundamental para la re-identificación visual porque reduce ruido de fondo y enfoca el análisis en el animal. Los crops generados serán usados para extraer embeddings en el Paso 03.

Los casos sin detección, con múltiples perros o con errores de crop no se eliminan automáticamente; se marcan como casos de revisión para mantener trazabilidad del pipeline.